# Background

Author: Diane Menuz  
Date: 03/02/2026   
Goal: Run UGS data through fluxqaqc and compare results with different inputs, such as NETRAD 1 vs.  
2 or different G values  
  
Notes  
- I used flags to set some values as np.nan and export different datasets depending on what flags I  
used
- You can also set up the config file to use your flag information to filter values, in which case  
you could probably use one dataset and different flagging specifications in the config file
- Must set up a separate config file for each run

# Data Filtering

**Flux-data-qaqc parameters**
- max_interp_hours: longest length gap to interpolate when Rn>0; default 2
- max_interp_hours_night: longest length gap to interpolate when Rn<0; default 4
- drop_gaps: whether to drop days when given percent of sub-daily data are missing; default True
- daily_frac: determine when to drop days. If set to 1, drop all days with any missing data. If set  
below 1, must have at least that proportion of data present or day will be dropped.

**From Volk 2023**  
- Uses same gap length filling as described above (using 2 for daytime and 4 for nighttime)
- States that the total number of subdaily gaps that were interpolated was recorded and used for  
post-filtering, but does not state what the cutoff value was.

# Setup

In [ ]:
station = 'US-UTD'
file = r'US-UTW_HH_202103101230_202510101030.csv'

In [ ]:
import pandas as pd
import numpy as np

from fluxdataqaqc import Data, QaQc, Plot
from bokeh.plotting import figure, show, ColumnDataSource
from bokeh.models.formatters import DatetimeTickFormatter
from bokeh.models import LinearAxis, Range1d
from bokeh.io import output_notebook
output_notebook()

In [ ]:
import sys
sys.path.append(r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/")
from micromet import validate
from micromet import data_cleaning
from micromet import eddy_plots as ed_plot
import calendar


# Functions

In [ ]:
def subset_year (df, 
                 year,
                 subset_type = 'growing_season',
                 gs_start = '04-01',
                 gs_end= f'10-31'):
    
    if subset_type == 'growing_season':
        mask = (df.index >= f'{year}-{gs_start} 0:00') & (df.index <= f'{year}-{gs_end} 23:30')
        output = df[mask]
    if subset_type == 'winter':
        mask = (df.index > f'{year}-{gs_end} 23:30') & (df.index < f'{year+1}-{gs_start} 0:00')
        output = df[mask]
    if output.empty:
        print(f"No data in {subset_type} for year {year}")
    else:
        print(f'Data subset from {output.index.min()} to {output.index.max()}')
        return(output)


# Read Data In, Impute Values, and Export

See file from Paul to see how data was imputed

# Run

In [ ]:
# set up parameters
year = 2025 # subset of data to look at
subset_type ='growing_season' # either winter or growing_season
variables = 'nr2_ga' # just for naming html file
comparison_name = f'{year}_{subset_type}_{variables}'

# config file name 
config_path = f'{station}_{variables}.ini'
print(comparison_name)
print(config_path)

In [ ]:
# read in config file and subset out data
d = Data(config_path)

d.df = subset_year(d.df, year)

print(f"Data file (time series) path: {d.climate_file}")
print(f'NETRAD var: {d.variables['Rn']}')
print(f'G var: {d.variables['G']}')

In [ ]:
# review data
d.plot(output_type='notebook', plot_width=700)

In [ ]:
# summarize to daily and monthtly; output html file
q = QaQc(d, daily_frac=1, max_interp_hours=2, max_interp_hours_night=4)

q.correct_data(meth='ebr', et_gap_fill=True)
ebr_gapfilled = q.df

q.plot(output_type='notebook', plot_width=700)
print(q.temporal_freq)
print(q.n_samples_per_day)
# saving the final files
q.plot(out_file=f'{station}_{comparison_name}.html')

In [ ]:
# monthly status
daily = q.df
daily['month'] = daily.index.month
daily[['month', 'ET_gap']].groupby('month').value_counts()

daily['data_status'] = 'data good'
mask = daily.ET_corr.isna()
daily.loc[mask, 'data_status'] = 'no ET data'
mask2 = daily.ET_gap == True
daily.loc[mask2, 'data_status'] = 'ET gap-filled'

pivot_df = daily.groupby('month')['data_status'].value_counts().unstack(fill_value=0)

# Optional: Ensure the columns are in a logical order
status_order = ['data good', 'ET gap-filled', 'no ET data']
pivot_df = pivot_df.reindex(columns=status_order)

print(pivot_df)


In [ ]:
# ed_plot.plotlystuff([d.df, d.df, d.df, d.df], 
#                     ['NETRAD_1_1_1_FINAL', 'LE_1_1_1', 'H_1_1_1', 'G_1_1_1'])